# 🎯 SCVD Violence Detection - Colab GUI
### خطوات بسيطة:
### 1️⃣ شغّل كل الـ Cells بالترتيب
### 2️⃣ ارفع ملف `best_model.pth` لما يطلب منك
### 3️⃣ ارفع أي فيديو وشوف النتيجة!

In [ ]:
# ============================================================
# CELL 1: تثبيت المكتبات
# ============================================================
!pip install -q gradio torch torchvision opencv-python-headless
print('✅ المكتبات جاهزة!')

In [ ]:
# ============================================================
# CELL 2: استورد المكتبات وجهّز الـ Model
# ============================================================
import torch
import torch.nn as nn
import cv2
import numpy as np
import gradio as gr
from torchvision import transforms, models
from google.colab import files

# ---- الإعدادات ----
CLASSES    = ['Normal', 'Violence', 'Weaponized']
NUM_FRAMES = 16
IMG_SIZE   = 112
COLORS     = {'Normal': '🟢', 'Violence': '🔴', 'Weaponized': '🟠'}
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ بيشتغل على: {device}')

# ---- الـ Transform ----
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ---- الـ Model ----
class CNNLSTMModel(nn.Module):
    def __init__(self):
        super(CNNLSTMModel, self).__init__()
        resnet          = models.resnet18(pretrained=False)
        self.cnn        = nn.Sequential(*list(resnet.children())[:-1])
        self.lstm       = nn.LSTM(512, 256, num_layers=2, batch_first=True, dropout=0.3)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.5), nn.Linear(128, 3)
        )
    def forward(self, x):
        b, t, C, H, W = x.shape
        x = x.view(b * t, C, H, W)
        with torch.no_grad():
            features = self.cnn(x)
        features    = features.view(b, t, -1)
        lstm_out, _ = self.lstm(features)
        return self.classifier(lstm_out[:, -1, :])

print('✅ الـ Model Architecture جاهز!')

In [ ]:
# ============================================================
# CELL 3: ارفع ملف best_model.pth
# ============================================================
print('📂 ارفع ملف best_model.pth ...')
uploaded = files.upload()

model = CNNLSTMModel().to(device)
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()
print('✅ الـ Model اتحمّل وجاهز!')

In [ ]:
# ============================================================
# CELL 4: الـ GUI - شغّله وارفع أي فيديو!
# ============================================================

def analyze_video(video_path):
    """بتاخد فيديو وبترجع التصنيف ونسب الثقة"""
    if video_path is None:
        return "❌ ارفع فيديو الأول!", {}

    try:
        cap         = cv2.VideoCapture(video_path)
        total       = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total == 0:
            return "❌ الفيديو مش شغّال، جرّب فيديو تاني", {}

        indices     = set([int(i * total / NUM_FRAMES) for i in range(NUM_FRAMES)])
        frames_list = []

        for i in range(total):
            ret, frame = cap.read()
            if not ret: break
            if i in indices:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames_list.append(transform(frame))
                if len(frames_list) == NUM_FRAMES: break
        cap.release()

        while len(frames_list) < NUM_FRAMES:
            frames_list.append(torch.zeros(3, IMG_SIZE, IMG_SIZE))

        tensor = torch.stack(frames_list).unsqueeze(0).to(device)
        with torch.no_grad():
            probs      = torch.softmax(model(tensor), dim=1)[0]
            pred_class = probs.argmax().item()

        label      = CLASSES[pred_class]
        confidence = probs[pred_class].item() * 100
        emoji      = COLORS[label]

        result_text = f"{emoji} {label}\n{confidence:.1f}% confidence"

        probs_dict = {
            f"{COLORS[cls]} {cls}": float(probs[i]) 
            for i, cls in enumerate(CLASSES)
        }

        return result_text, probs_dict

    except Exception as e:
        return f"❌ Error: {str(e)}", {}


# ---- بناء الـ GUI ----
with gr.Blocks(
    title="SCVD Violence Detector",
    theme=gr.themes.Base(
        primary_hue="blue",
        neutral_hue="slate"
    )
) as demo:

    gr.HTML("""
    <div style='text-align:center; padding: 20px; background: linear-gradient(135deg, #0A1628, #1A3A5C); border-radius: 12px; margin-bottom: 20px;'>
        <h1 style='color: #00C2FF; font-size: 2em; margin: 0;'>🎯 SCVD Violence Detection</h1>
        <p style='color: #AACCEE; margin: 8px 0 0 0;'>Smart-City CCTV Violence Detection System | CNN + LSTM</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📹 ارفع الفيديو هنا")
            video_input = gr.Video(
                label="Video Input",
                height=300
            )
            analyze_btn = gr.Button(
                "🔍 حلّل الفيديو",
                variant="primary",
                size="lg"
            )

        with gr.Column(scale=1):
            gr.Markdown("### 📊 النتيجة")
            result_output = gr.Textbox(
                label="التصنيف",
                lines=3,
                text_align="center",
                elem_id="result-box"
            )
            probs_output = gr.Label(
                label="نسب الثقة لكل Class",
                num_top_classes=3
            )

    gr.HTML("""
    <div style='text-align:center; margin-top: 15px; padding: 12px; background: #132038; border-radius: 8px;'>
        <span style='color:#2ED573;'>🟢 Normal</span> &nbsp;&nbsp;
        <span style='color:#FF4757;'>🔴 Violence</span> &nbsp;&nbsp;
        <span style='color:#FFA500;'>🟠 Weaponized</span>
        <p style='color:#667788; font-size:0.85em; margin: 6px 0 0 0;'>Model: ResNet18 + LSTM | Dataset: SCVD | Accuracy: 99.8%</p>
    </div>
    """)

    analyze_btn.click(
        fn=analyze_video,
        inputs=[video_input],
        outputs=[result_output, probs_output]
    )

demo.launch(share=True, debug=False)
print('✅ الـ GUI شغّال! افتح الـ Link فوق')